## Синтетическая разметка новостного корпуса lenta-ru через DeepPavlov

In [1]:
import pandas as pd
from tqdm import tqdm
from corus import load_lenta
from deeppavlov import build_model
from transformers import AutoTokenizer, logging
import urllib.request
import warnings

logging.set_verbosity_error()
warnings.filterwarnings("ignore", message=".*resume_download.*", category=FutureWarning, module="huggingface_hub.file_download"
                        )
# Фиксируем для воспроизводимости
RANDOM_SEED = 12

In [2]:
try:
    with open("lenta-ru-news.csv.gz", "rb"):
        pass
except FileNotFoundError:
    print("Downloading dataset...")
    urllib.request.urlretrieve("https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz", "lenta-ru-news.csv.gz")
    print("Download completed.")

In [3]:
model = build_model('ner_collection3_bert', download=True, install=True)

2025-04-16 12:24:10.762 INFO in 'deeppavlov.download'['download'] at line 138: Skipped http://files.deeppavlov.ai/v1/ner/ner_rus_bert_coll3_torch.tar.gz download because of matching hashes


In [4]:
# Загружаем корпус новостей и готовим DataFrame
raw_news = load_lenta('data/lenta-ru-news.csv.gz')
news_df = pd.DataFrame(raw_news, columns=['url', 'title', 'text', 'topic', 'tags', 'date'])
news_df = news_df.sample(n=10000, random_state=RANDOM_SEED)[['text']].reset_index(drop=True)
news_df.head()

,text
0,"В афганской провинции Нангархар, находящейся н..."
1,Отток капитала из России по итогам первых четы...
2,Эдриану Броуди предложили роль в триллере Альб...
3,Во вторник в Красноярск прибыла съемочная груп...
4,В четверг три жителя Тегерана были подвергнуты...


In [5]:
print(model([news_df["text"].iloc[10]]))

[[['Представители', 'финансовой', 'полиции', 'Грузии', 'опечатали', 'супермаркет', 'и', 'ресторан', 'в', 'городе', 'Поти', ',', 'основным', 'собственником', 'которых', ',', 'по', 'их', 'данным', ',', 'является', 'бывший', 'представитель', 'президента', 'в', 'Кодорском', 'ущелье', 'Эмзар', 'Квициани', ',', 'сообщает', 'грузинский', 'телеканал', '"', 'Рустави', '2', '"', '.Полиция', 'изъяла', 'бухгалтерские', 'документы', 'компании', '"', 'Vardis', 'ubani', '"', 'Ltd.', ',', 'на', 'которую', 'оформлены', 'магазин', 'и', 'ресторан.', 'Компанию', 'подозревают', 'в', 'уклонении', 'от', 'уплаты', 'налогов.Между', 'тем', 'в', 'селе', 'Чхалта', 'Кодорского', 'ущелья', 'проводится', 'экспертиза', 'оружия', ',', 'найденного', 'в', 'доме', 'Эмзара', 'Квициани', ',', 'сообщает', 'ИА', '"', 'Новости', '-', 'Грузия', '"', '.', 'Помимо', 'стрелкового', 'оружия', 'полицейские', 'обнаружили', 'переносной', 'зенитно', '-', 'ракетный', 'комплекс', '"', 'Стрела', '"', 'и', 'гранатомет', '"', 'Муха', '"', 

In [6]:
def truncate_texts_by_token_count(texts, model_identifier, max_tokens):
    tokenizer = AutoTokenizer.from_pretrained(model_identifier)
    truncated_texts = []
    for text in texts:
        token_list = tokenizer.tokenize(text)
        if len(token_list) > max_tokens:
            token_list = token_list[:max_tokens]
            text = tokenizer.convert_tokens_to_string(token_list)
        truncated_texts.append(text)
    return truncated_texts

In [ ]:
# обрезаем тексты до 400 токенов
valid_texts = truncate_texts_by_token_count(news_df['text'], "DeepPavlov/rubert-base-cased", 400)
print(len(valid_texts))

synthetic_annotations = [model([txt]) for txt in tqdm(valid_texts)]
print(synthetic_annotations[0]) 

10000


100%|██████████| 10000/10000 [05:03<00:00, 32.96it/s]

[[['В', 'афганской', 'провинции', 'Нангархар', ',', 'находящейся', 'на', 'востоке', 'страны', ',', 'группа', 'боевиков', 'террористической', 'организации', '«', 'Исламское', 'государство', '»', '(', 'ИГ', ')', 'подорвалась', 'при', 'транспортировке', 'своей', 'взрывчатки.', 'По', 'меньшей', 'мере', '12', 'экстремистов', 'погибли', ',', 'еще', '21', 'получил', 'ранения.', 'Об', 'этом', 'Khaama', 'Press', 'сообщает', 'со', 'ссылкой', 'на', 'Министерство', 'обороны', 'Афганистана.', 'Боевики', 'пытались', 'доставить', 'взрывчатое', 'вещество', 'из', 'района', 'Маздак', 'на', 'рынок', 'Шедил', ',', 'когда', 'устройство', 'самопроизвольно', 'сдетонировало.', 'О', 'пострадавших', 'мирных', 'жителях', 'не', 'сообщается.', 'В', 'последние', 'месяцы', 'ситуация', 'в', 'Афганистане', 'заметно', 'ухудшилась.', 'На', 'территории', 'страны', 'выросло', 'влияние', 'ИГ.', 'Отряды', 'действующих', 'в', 'стране', 'террористов', '«', 'Исламского', 'государства', '»', 'в', 'основном', 'представляют', 'со

In [8]:
#  словарь для преобразования тегов из S/E формата в стандартные BIO-теги
replacement = {
    'S-LOC': 'B-LOC',
    'E-LOC': 'I-LOC',
    'S-ORG': 'B-ORG',
    'E-ORG': 'I-ORG',
    'S-PER': 'B-PER',
    'E-PER': 'I-PER'
}

def remap_annotation(annot, mapping):
    new_tags = [mapping.get(tag, tag) for tag in annot[1][0]]
    return [annot[0], [new_tags]]

remapped_annots = [remap_annotation(a, replacement) for a in synthetic_annotations]

In [9]:
# уникальные BIO-теги
bio_tags_set = {tag for a in remapped_annots for tag in a[1][0]}
bio_tags_set

{'B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O'}

In [10]:
final_df = pd.DataFrame({
    'text': valid_texts,
    'annotation': remapped_annots
})

final_df.to_parquet('lenta_data.parquet', index=False)